In [ ]:
# Setup: imports
import pandas as pd
from pathlib import Path

# Configuration
DATA_CSV = Path('../dataset_filtered/articles.csv')  # relative to notebook location
OUT_DIR = Path('../dataset_filtered/articles_by_product_type')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Reading: {DATA_CSV.resolve()}')
df = pd.read_csv(DATA_CSV)
print('Rows:', len(df), 'Columns:', len(df.columns))
print('Columns list:', df.columns.tolist()[:20], '...')
print(df.head(3))

# Make Parquet Partitions

This notebook:

1. Loads `articles.csv` from `dataset_filtered`.
2. Explores the distribution of `product_type_name`.
3. Writes one Parquet file per `product_type_name` into `articles_by_product_type`.
4. Optionally writes a directory-partitioned Parquet dataset for efficient downstream reads.

Outputs:
- `../dataset_filtered/articles_by_product_type/*.parquet`
- `../dataset_filtered/articles_parquet_partitioned/product_type_name=.../part.parquet`

Adjust paths if running from a different working directory.

In [ ]:
# Value counts for product_type_name to understand distribution
key_col = 'product_type_name'
if key_col not in df.columns:
    raise KeyError(f'Missing expected column {key_col} in dataframe')

counts = df[key_col].value_counts().sort_values(ascending=False)
print(counts.head(10))
print(f'Total distinct {key_col}:', counts.shape[0])

In [ ]:
# Write one parquet file per product_type_name (sanitized file names)
from datetime import datetime
import re

written = []
for product_type, group in df.groupby(key_col):
    safe_name = re.sub(r'[^A-Za-z0-9_.-]+', '_', product_type.strip())[:120].strip('_')
    out_path = OUT_DIR / f'{safe_name}.parquet'
    group.to_parquet(out_path, index=False)
    written.append((product_type, len(group), out_path.name))

print(f'Wrote {len(written)} parquet files into {OUT_DIR.resolve()}')
print('Sample:', written[:5])

In [ ]:
# OPTIONAL: Write a single partitioned parquet dataset (directory structure) for efficient loading
# This creates OUT_DIR_parent/product_type_name=<value>/part.parquet layout
partition_root = OUT_DIR.parent / 'articles_parquet_partitioned'
partition_root.mkdir(parents=True, exist_ok=True)

# Use pandas to_parquet with partition_cols (pandas >= 2.0 with pyarrow engine)
try:
    df.to_parquet(partition_root, partition_cols=[key_col], engine='pyarrow', index=False)
    print('Partitioned dataset written at', partition_root.resolve())
except Exception as e:
    print('Partitioned write failed (possibly due to pandas/pyarrow version) ->', e)